In [3]:
import numpy as np
from sage.all import *

# Deletion on e
def left_operation(F, e):
    F.delete_edge(e)
    return F

# Contraction on e plus singleton
def middle_operation(F, e):
    F.contract_edge(e)
    F.add_vertex()
    return F

# Contraction on e plus singleton and edge
def right_operation(F, e):
    F.contract_edge(e)
    F.add_vertex()
    F.add_edge(e)
    return F

In [4]:
# If tree has an internal edge, it returns one; otherwise, returns None
def get_internal_edge(F):
    for e in F.edges(labels=False):
        if F.degree(e)[0] > 1 and F.degree(e)[1] > 1:
            return e
            break
    return None

# Get list of internal egdes
def list_internal_edges(T):
    list = []
    for e in T.edges(labels=False):
        if T.degree(e)[0] > 1 and T.degree(e)[1] > 1:
            list.append(e)
    return list

# Get leading partition from the tree itself
def leading_partition(T):
    internal_edges = list_internal_edges(T)
    
    for e in internal_edges:
        T.delete_edge(e)
        
    return T.connected_components_sizes()

# Return number of internal edges from the tree
def number_internal_edges(tree):
    count = 0
    for e in tree.edges(labels=False):
        if (tree.degree(e)[0] > 1) and (tree.degree(e)[1] > 1):
            count = count + 1
    return count

def string_sign(x):
    sign = ""
    if x>0:
        sign = sign + "+"
    else:
        sign = sign + "-"

    return sign

# Input: array p
# Output: returns the number of 1's in p
def num_singletons(p):
    num = 0
    for i in p:
        if i == 1:
            num += 1
    return num

# Input: an integer partition p, the set of partitions of n
# Output: returns the index of p in Partitions(n), assuming lexicographic ordering and zero-indexing
def get_index(p, partitions):
    i = 0
    for partition in partitions:
        if partition == p:
            return int(i)
        else:
            i += 1

# Input: a CSF vector, and the number of vertices n           
# Output: returns the leading partion 
def get_leading_partition(vector, n):
    partitions = Partitions(n).list()
    leading = 0 
    count = 0
    
    while count < len(partitions):
        if vector[count] != 0:
            leading = vector[count]
            last_index = count
        count += 1

    return partitions[last_index]

def create_tree(vertex_list, edge_list):
    G = Graph()
    for i in range(0, len(vertex_list)):
        G.add_vertex()
    for edge in edge_list:
        G.add_edge(edge)
        
    # G.plot().show()
    return G

In [5]:
# Input: A forest F on n vertices, a list of forests seen so far
# Output: Returns the CSF of F as a vector of length p(n), in lexicographic order
def CSF_helper(F, n, seen_list):
    partitions = Partitions(n)
    # Base case: F has already been seen
    for seen_forest in seen_list:
        if seen_forest.is_isomorphic(F):
            return seen_list[seen_forest]

    CSF = [0] * len(partitions)
    # Base case: F has no internal edges
    e = get_internal_edge(F)
    if e == None:
        p = F.connected_components_sizes()
        sign = (-1)**num_singletons(p)
        index = get_index(p, partitions)
        CSF[index] = sign
        seen_list[F.copy(immutable=True)] = CSF
        return CSF

    # Recursive case: F has not been seen and has an internal edge
    else: 
        F1 = F.copy()
        F2 = F.copy()
        F3 = F.copy()
        CSF = np.add(CSF_helper(left_operation(F1,e),n,seen_list), CSF_helper(middle_operation(F2,e),n,seen_list))
        CSF = np.add(CSF, CSF_helper(right_operation(F3,e),n,seen_list))
        seen_list[F.copy(immutable=True)] = CSF
        return CSF

In [8]:
# DNC for al trees in n vertices, also filter by number of internal edges if necessary
# Input: number of vertices, number of internal edges, diameter. 
#        If we do not want to restrict by number of internal edges, input zero.
# One can ommit diameter if not necessary

def CSF_iterator(n, int_edges, diameter=0):
    tree_list = []
    partitions_list = Partitions(n).list()
    seen_list = {}
    tree_iterator = graphs.trees(n)
    
    for T in tree_iterator:
        tree_list.append(T)            
        
    L_T = len(tree_list)
    L_P = len(partitions_list)
    
    if diameter == 0:
        if int_edges == 0:
            for i in range(L_T):
                tree_list[i] = tree_list[i].copy()
                CSF_vector = CSF_helper(tree_list[i], n, seen_list)
                l_p = get_leading_partition(CSF_vector, n)
                
                tree_list[i].plot().show()
                print("Leading partition:", l_p)

                vector_partitions = partitions_in_tree(CSF_vector, n)
                print(coefficient_printer(CSF_vector, vector_partitions))
                        
        else:
            for i in range(L_T):
                if number_internal_edges(tree_list[i]) == int_edges:
                    tree_list[i] = tree_list[i].copy()
                    CSF_vector = CSF_helper(tree_list[i], n, seen_list)
                    l_p = get_leading_partition(CSF_vector, n)

                    tree_list[i].plot().show()
                    print("Leading partition:", l_p)

                    vector_partitions = partitions_in_tree(CSF_vector, n)
                    print(coefficient_printer(CSF_vector, vector_partitions))

    else:
        if int_edges == 0:
            for i in range(L_T):
                if tree_list[i].diameter() == diameter:
                    tree_list[i] = tree_list[i].copy()
                    tree_list[i].plot().show()

                    CSF_vector = CSF_helper(tree_list[i], n, seen_list)
                    l_p = get_leading_partition(CSF_vector, n)
                    
                    print("Leading partition:", l_p)
                    
                    vector_partitions = partitions_in_tree(CSF_vector, n)
                    print(coefficient_printer(CSF_vector, vector_partitions))
        else:
            for i in range(L_T):
                if tree_list[i].diameter() == diameter and number_internal_edges(tree_list[i]) == int_edges:
                    tree_list[i] = tree_list[i].copy()
                    tree_list[i].plot().show()

                    CSF_vector = CSF_helper(tree_list[i], n, seen_list)
                    
                    l_p = get_leading_partition(CSF_vector, n)
                    print("Tree index:", len(M))
                    
                    print("Leading partition:", l_p)
                    
                    vector_partitions = partitions_in_tree(CSF_vector, n)
                    print(coefficient_printer(CSF_vector, vector_partitions))

In [7]:
# DNC for a tree given the vertex and edge set
# Input: two arrays, one with the labeling of the vertices starting at zero, e.g. [0,1,2,3,4,5] for a tree on 6 vertices
# The other array are the edges, e.g. [(0,1), (1,2), (2,3), (4,5)]

def CSF_edgeSet(vertex_list, edge_list):
    length_list = len(vertex_list)
    leading = 0 
    partitions_list = Partitions(length_list).list()
    count = 0
    
    partitions = Partitions(len(vertex_list))
    seen_list = {}
    
    tree = create_tree(vertex_list, edge_list)
    tree.plot().show()
    tree = tree.copy()
    
    CSF_vector = CSF_helper(tree, len(vertex_list), seen_list)
    l_p = get_leading_partition(CSF_vector, len(vertex_list))
    
    print("Leading partition:", l_p)
                
    vector_partitions = partitions_in_tree(CSF_vector, len(vertex_list))
    print(coefficient_printer(CSF_vector, vector_partitions))
    
    return CSF_vector

In [ ]:
# DNC for a tree given an object of the class Graph
# Input: a tree as an object of the class Graph in sagemath
def CSF_tree(tree):
    vertex_list = tree.vertices()
    edge_list = tree.edges(labels=False)
    length_list = len(vertex_list)
    partitions_list = Partitions(length_list).list()
    
    partitions = Partitions(len(vertex_list))
    seen_list = {}
    

    tree.plot().show()
    tree = tree.copy()
    
    CSF_vector = CSF_helper(tree, len(vertex_list), seen_list)
    l_p = get_leading_partition(CSF_vector, len(vertex_list))
    
    print("Leading partition:", l_p)
                
        
    vector_partitions = partitions_in_tree(CSF_vector, len(vertex_list))
    print(coefficient_printer(CSF_vector, vector_partitions))
    
    return CSF_vector